In [ ]:
# --- 1. TELEPÍTÉS ---
!pip install -q vllm pyngrok

import os
import subprocess
import time
import sys
from pyngrok import ngrok

# --- 2. KONFIGURÁCIÓ ---
MODEL_ID = "Qwen/Qwen2.5-14B-Instruct-AWQ"
AUTH_TOKEN = "2sUqYYhi9BuVxa2zJtzAdHVI9eS_6rxXaeqe4o8JTPtMPtsBf"

# --- 3. TAKARÍTÁS ---
print("🧹 Takarítás...")
os.system("pkill -f vllm")
os.system("pkill -f ngrok")
ngrok.kill()

# --- 4. NGROK ---
if not AUTH_TOKEN:
    print("❌ HIBA: Nincs AUTH_TOKEN!")
else:
    ngrok.set_auth_token(AUTH_TOKEN)
    try:
        tunnel = ngrok.connect(8000, domain="intimate-polecat-adjusted.ngrok-free.app")
        print(f"✅ Fix domain aktív!")
    except Exception as e:
        print(f"⚠️ Fix domain nem sikerült, random URL kérése...")
        tunnel = ngrok.connect(8000)

    PUBLIC_URL = tunnel.public_url
    print(f"🎉 API LESZ ITT: {PUBLIC_URL}/v1")

# --- 5. vLLM SZERVER INDÍTÁSA (JAVÍTOTT PARANCSOKKAL) ---
log_file = open("vllm_server.log", "w")

cmd = [
    "python",
    "-m",
    "vllm.entrypoints.openai.api_server",
    "--model",
    MODEL_ID,
    "--quantization",
    "awq",
    "--dtype",
    "half",
    "--host",
    "0.0.0.0",
    "--port",
    "8000",
    # Memória és Stabilitás
    "--max-model-len",
    "4096",
    "--gpu-memory-utilization",
    "0.95",
    "--enforce-eager",
    "--trust-remote-code",
    # --- EZ A JAVÍTÁS A ROO CODE-HOZ! ---
    "--enable-auto-tool-choice",  # Engedélyezi, hogy a Roo Code kérje az eszközöket
    "--tool-call-parser",
    "hermes",  # A Qwen/Llama modellekhez ez a parser vált be legjobban
]

print(f"\n🚀 Szerver indítása (Tool support bekapcsolva)...")
process = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)

# --- 6. LOG MONITOROZÁS ---
try:
    print("⏳ Várakozás a modell betöltésére (kb. 2-3 perc)...")
    with open("vllm_server.log", "r") as f:
        f.seek(0, 2)
        while True:
            line = f.readline()
            if line:
                print(line.strip())
                if "Uvicorn running on" in line:
                    print("\n✅✅✅ A SZERVER SIKERESEN ELINDULT! ✅✅✅")
                    print("Most próbáld újra a Roo Code-ban a kérést!")
                    break
            else:
                time.sleep(0.5)
                if process.poll() is not None:
                    print("\n❌ A SZERVER LEÁLLT HIBÁVAL!")
                    break

    if process.poll() is None:
        process.wait()

except KeyboardInterrupt:
    print("\n🛑 Leállítás...")
    process.terminate()
    ngrok.kill()
    log_file.close()